# QGECAF reviewer-revision run
This runner imports the corrected nested-BGWO implementation. Select the `chaitalienv` kernel and verify the paths before training.

In [1]:
import sys
import tensorflow as tf

print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

Python executable: C:\Users\viraj\anaconda3\envs\chaitalienv\python.exe
TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [1]:
from pathlib import Path
from QGECAF_Reviewer_Revision_Nested_BGWO import Config, main

cfg = Config(
    dataset=r"D:/chaitali/patientsplit801010_clean_v2",

    metadata_csv=(
        r"D:/chaitali/patientsplit801010_clean_v2/"
        r"patient_split_80_10_10.csv"
    ),

    locked_manifest=(
        r"D:/chaitali/qgacef/QGECAF_FROM_SCRATCH_RUN_01/"
        r"locked_patient_fold_assignments.csv"
    ),

    output=(
        r"D:/chaitali/qgacef/"
        r"QGECAF_REVIEWER_REVISION_NESTED_BGWO"
    ),

    resume=True,
    run_end_to_end_latency=True,
)

In [2]:
from pathlib import Path
from QGECAF_Reviewer_Revision_Nested_BGWO import Config, main

cfg = Config(
    dataset=r"D:/chaitali/patientsplit801010_clean_v2",
    metadata_csv=r"D:/chaitali/patientsplit801010_clean_v2/patient_split_80_10_10.csv",
    locked_manifest=r"D:/chaitali/qgacef/QGECAF_FROM_SCRATCH_RUN_01/locked_patient_fold_assignments.csv",
    output=r"D:/chaitali/qgacef/QGECAF_REVIEWER_REVISION_NESTED_BGWO",
    resume=True,
    run_end_to_end_latency=True,
)

In [3]:
for label, value in {
    'dataset': cfg.dataset,
    'metadata': cfg.metadata_csv,
    'locked manifest': cfg.locked_manifest,
}.items():
    print(f"{label}: {value} -> {Path(value).exists()}")
print('output:', cfg.output)

dataset: D:/chaitali/patientsplit801010_clean_v2 -> True
metadata: D:/chaitali/patientsplit801010_clean_v2/patient_split_80_10_10.csv -> True
locked manifest: D:/chaitali/qgacef/QGECAF_FROM_SCRATCH_RUN_01/locked_patient_fold_assignments.csv -> True
output: D:/chaitali/qgacef/QGECAF_REVIEWER_REVISION_NESTED_BGWO


In [4]:
import pandas as pd
from pathlib import Path

manifest = pd.read_csv(Path(cfg.locked_manifest))

print("Columns:", manifest.columns.tolist())
print("Outer-fold values:", sorted(manifest["outer_fold"].unique()))
print("\nImages per fold:")
print(manifest["outer_fold"].value_counts().sort_index())

print("\nPatients per fold:")
print(
    manifest.groupby("outer_fold")["patient_id"]
    .nunique()
    .sort_index()
)

# Confirm one patient belongs to only one outer fold
patient_fold_counts = (
    manifest.groupby("patient_id")["outer_fold"]
    .nunique()
)

problem_patients = patient_fold_counts[
    patient_fold_counts > 1
]

print("\nPatients assigned to multiple folds:", len(problem_patients))

assert len(problem_patients) == 0
assert -1 in manifest["outer_fold"].unique()

print("\nPASS: Locked patient-fold manifest is valid.")

Columns: ['path', 'patient_id', 'label', 'class_name', 'original_split', 'outer_fold']
Outer-fold values: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Images per fold:
-1     1135
 1     1011
 2     1123
 3     1252
 4      941
 5     1139
 6     1076
 7     1279
 8     1100
 9     1184
 10    1160
Name: outer_fold, dtype: int64

Patients per fold:
outer_fold
-1     180
 1     158
 2     164
 3     168
 4     163
 5     166
 6     157
 7     161
 8     157
 9     161
 10    157
Name: patient_id, dtype: int64

Patients assigned to multiple folds: 0

PASS: Locked patient-fold manifest is valid.


In [5]:
development = manifest[manifest["outer_fold"] != -1]
test = manifest[manifest["outer_fold"] == -1]

development_patients = set(
    development["patient_id"].astype(str)
)

test_patients = set(
    test["patient_id"].astype(str)
)

overlap = development_patients.intersection(
    test_patients
)

print("Development images:", len(development))
print("Development patients:", len(development_patients))
print("Test images:", len(test))
print("Test patients:", len(test_patients))
print("Patient overlap:", overlap)

assert len(overlap) == 0

print("PASS: No development–test patient leakage.")

Development images: 11265
Development patients: 1612
Test images: 1135
Test patients: 180
Patient overlap: set()
PASS: No development–test patient leakage.


main(cfg)

In [6]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(cfg.output)
FINAL_DIR = ROOT / "final_untouched_test"
LATENCY_DIR = FINAL_DIR / "raw_image_to_prediction_benchmark"

print("Run complete:", (ROOT / "RUN_COMPLETE.txt").exists())

print("\nAvailable principal files:")
for file_path in [
    ROOT / "fold_metrics.csv",
    ROOT / "cross_validation_summary.csv",
    ROOT / "final_test_metrics.csv",
    FINAL_DIR / "model_complexity.json",
    FINAL_DIR / "final_training_protocol.json",
    LATENCY_DIR / "end_to_end_latency_summary.csv",
]:
    print(file_path.name, "->", file_path.exists())

if (ROOT / "cross_validation_summary.csv").exists():
    print("\n10-fold cross-validation summary")
    display(pd.read_csv(ROOT / "cross_validation_summary.csv"))

if (ROOT / "final_test_metrics.csv").exists():
    print("\nUntouched test results")
    display(pd.read_csv(ROOT / "final_test_metrics.csv").T)

if (FINAL_DIR / "model_complexity.json").exists():
    print("\nModel complexity")
    complexity = json.loads(
        (FINAL_DIR / "model_complexity.json").read_text()
    )
    display(pd.Series(complexity, name="Value").to_frame())

if (FINAL_DIR / "final_training_protocol.json").exists():
    print("\nFinal training information")
    training = json.loads(
        (FINAL_DIR / "final_training_protocol.json").read_text()
    )
    display(pd.Series(training, name="Value").to_frame())

if (LATENCY_DIR / "end_to_end_latency_summary.csv").exists():
    print("\nRaw-image-to-prediction latency")
    display(
        pd.read_csv(
            LATENCY_DIR / "end_to_end_latency_summary.csv"
        )
    )

Run complete: True

Available principal files:
fold_metrics.csv -> True
cross_validation_summary.csv -> True
final_test_metrics.csv -> True
model_complexity.json -> True
final_training_protocol.json -> True
end_to_end_latency_summary.csv -> True

10-fold cross-validation summary


,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,accuracy,10,0.925976,0.014297,0.915748,0.936203
1,balanced_accuracy,10,0.935963,0.013370,0.926399,0.945527
2,macro_f1,10,0.911895,0.014578,0.901467,0.922324
3,weighted_f1,10,0.926756,0.014382,0.916468,0.937044
4,mcc,10,0.906647,0.016698,0.894701,0.918592
5,ece_15_bins,10,0.029862,0.009194,0.023285,0.036439
6,brier_multiclass,10,0.121129,0.020789,0.106257,0.136001
7,negative_log_likelihood,10,0.298073,0.044607,0.266164,0.329983
8,uncertainty_error_auroc,10,0.746023,0.053141,0.708008,0.784038
9,quality_uncertainty_spearman_rho,10,-0.038065,0.049162,-0.073233,-0.002897



Untouched test results


,0
evaluation,untouched_test
accuracy,0.948018
balanced_accuracy,0.926219
macro_f1,0.926153
weighted_f1,0.947303
mcc,0.933278
ece_15_bins,0.039158
brier_multiclass,0.091581
negative_log_likelihood,0.222198
uncertainty_error_auroc,0.810724



Model complexity


,Value
total_parameters,2423623
trainable_parameters,1692039
nontrainable_parameters,731584
total_parameters_million,2.423623
network_flops_per_image,198367821
network_gflops_per_image,0.198368
approximate_gmacs_per_image,0.099184
batch_size_used_for_profiling,1
float_precision_assumed_for_memory,float32
weight_memory_mib,9.245388



Final training information


,Value
selected_epochs,11
selection_rule,rounded median of the 10 fold-specific best va...
development_images,11265
development_patients,1612
test_images,1135
test_patients,180
test_used_during_model_selection,False
training_time_seconds,454.096722
mean_epoch_time_seconds,41.113451



Raw-image-to-prediction latency


,batch_size,repetitions,mean_ms_per_image,sample_SD_ms_per_image,median_ms_per_image,iqr_ms_per_image,p95_ms_per_image,mean_fps
0,1,5,198.007686,7.499806,194.373276,1.401148,208.252953,5.055836
1,8,5,95.910853,7.018127,96.318143,9.711608,102.737350,10.472404
2,16,5,423.810307,740.329092,93.336479,1.072350,1417.344258,8.744019


In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path(cfg.output)

raw_latency_file = (
    ROOT
    / "final_untouched_test"
    / "raw_image_to_prediction_benchmark"
    / "end_to_end_latency_raw.csv"
)

latency_raw = pd.read_csv(raw_latency_file)

stage_columns = [
    "image_loading_seconds",
    "descriptors_quality_seconds",
    "transformations_seconds",
    "model_forward_seconds",
    "postprocessing_seconds",
    "raw_image_to_prediction_seconds",
]

for column in stage_columns:
    latency_raw[column.replace("_seconds", "_ms_per_image")] = (
        latency_raw[column]
        * 1000
        / latency_raw["images"]
    )

display(
    latency_raw[
        latency_raw["batch_size"] == 16
    ].sort_values(
        "raw_image_to_prediction_ms_per_image",
        ascending=False,
    )
)

,batch_size,repetition,images,image_loading_seconds,descriptors_quality_seconds,transformations_seconds,model_forward_seconds,postprocessing_seconds,raw_image_to_prediction_seconds,raw_image_to_prediction_ms_per_image,raw_image_to_prediction_fps,image_loading_ms_per_image,descriptors_quality_ms_per_image,transformations_ms_per_image,model_forward_ms_per_image,postprocessing_ms_per_image
13,16,4,1135,10.200811,1511.480273,0.299941,462.098969,0.005653,1984.148955,1748.148859,0.572034,8.987499,1331.700681,0.264265,407.135656,0.004980
10,16,1,1135,9.704590,85.689163,0.322918,11.042045,0.005967,106.832847,94.125856,10.624073,8.550300,75.497060,0.284509,9.728674,0.005257
11,16,2,1135,9.817144,84.736111,0.318307,10.991657,0.006650,105.936904,93.336479,10.713925,8.649466,74.657366,0.280447,9.684280,0.005859
12,16,3,1135,9.713581,84.637745,0.319185,10.871617,0.005970,105.615730,93.053507,10.746505,8.558221,74.570701,0.281220,9.578518,0.005260
14,16,5,1135,12.952936,78.899794,0.287135,10.384207,0.006256,102.589059,90.386836,11.063558,11.412278,69.515237,0.252982,9.149081,0.005512


In [8]:
from pathlib import Path
import re

# ============================================================
# 1. FINAL MODEL SUMMARY PATH
# ============================================================

SUMMARY_FILE = Path(
    r"D:\chaitali\qgacef"
    r"\QGECAF_REVIEWER_REVISION_NESTED_BGWO"
    r"\final_untouched_test"
    r"\final_model_summary.txt"
)

print("=" * 80)
print("FINAL MODEL SUMMARY CHECK")
print("=" * 80)
print("File:", SUMMARY_FILE)
print("Exists:", SUMMARY_FILE.is_file())

if not SUMMARY_FILE.is_file():
    raise FileNotFoundError(
        f"Model-summary file was not found:\n{SUMMARY_FILE}"
    )

# ============================================================
# 2. READ AND DISPLAY THE SUMMARY
# ============================================================

summary_text = SUMMARY_FILE.read_text(
    encoding="utf-8",
    errors="replace"
)

print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)
print(summary_text)

# ============================================================
# 3. IDENTIFY THE MOBILENET BACKBONE
# ============================================================

summary_lower = summary_text.lower()

mobilenet_v2_patterns = [
    "mobilenetv2",
    "mobilenet_v2",
    "block_1_expand",
    "block_1_project",
    "expanded_conv",
    "inverted_residual",
]

mobilenet_v1_patterns = [
    "mobilenetv1",
    "mobilenet_v1",
    "conv_pw_",
    "conv_dw_",
    "depthwise_conv2d",
]

v2_matches = [
    pattern for pattern in mobilenet_v2_patterns
    if pattern in summary_lower
]

v1_matches = [
    pattern for pattern in mobilenet_v1_patterns
    if pattern in summary_lower
]

print("\n" + "=" * 80)
print("BACKBONE IDENTIFICATION")
print("=" * 80)

if v2_matches and not v1_matches:
    detected_backbone = "MobileNetV2"
elif v1_matches and not v2_matches:
    detected_backbone = "MobileNetV1"
elif v2_matches and v1_matches:
    # Generic depthwise layer names can occur in multiple architectures.
    if any(
        pattern in summary_lower
        for pattern in [
            "mobilenetv2",
            "mobilenet_v2",
            "block_1_expand",
            "block_1_project",
            "expanded_conv",
        ]
    ):
        detected_backbone = "MobileNetV2"
    elif any(
        pattern in summary_lower
        for pattern in [
            "mobilenetv1",
            "mobilenet_v1",
            "conv_pw_",
            "conv_dw_",
        ]
    ):
        detected_backbone = "MobileNetV1"
    else:
        detected_backbone = "Ambiguous"
else:
    detected_backbone = "Not identified from saved summary"

print("Detected backbone:", detected_backbone)
print("MobileNetV2 indicators:", v2_matches or "None")
print("MobileNetV1 indicators:", v1_matches or "None")

# ============================================================
# 4. EXTRACT PARAMETER COUNTS
# ============================================================

parameter_patterns = {
    "Total parameters": r"Total params:\s*([\d,]+)",
    "Trainable parameters": r"Trainable params:\s*([\d,]+)",
    "Non-trainable parameters": r"Non-trainable params:\s*([\d,]+)",
}

print("\n" + "=" * 80)
print("PARAMETER COUNTS")
print("=" * 80)

parameter_results = {}

for name, pattern in parameter_patterns.items():
    match = re.search(pattern, summary_text, flags=re.IGNORECASE)

    if match:
        value = int(match.group(1).replace(",", ""))
        parameter_results[name] = value
        print(f"{name:27s}: {value:,}")
    else:
        parameter_results[name] = None
        print(f"{name:27s}: Not found in summary")

# Verify parameter accounting
total = parameter_results["Total parameters"]
trainable = parameter_results["Trainable parameters"]
nontrainable = parameter_results["Non-trainable parameters"]

if all(value is not None for value in [total, trainable, nontrainable]):
    calculated_total = trainable + nontrainable

    print("\nParameter-accounting check:")
    print(f"Trainable + non-trainable: {calculated_total:,}")
    print(f"Reported total:           {total:,}")

    if calculated_total == total:
        print("PASS: Parameter counts are internally consistent.")
    else:
        print(
            "WARNING: Parameter counts differ by "
            f"{total - calculated_total:,} parameters."
        )

print("\nRecommended matched benchmark:", detected_backbone)

FINAL MODEL SUMMARY CHECK
File: D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\final_untouched_test\final_model_summary.txt
Exists: True

MODEL SUMMARY
Model: "QGECAF_MaternalFetal"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image (InputLayer)             [(None, 128, 128, 1  0           []                               
                                )]                                                                
                                                                                                  
 grayscale_to_rgb (Concatenate)  (None, 128, 128, 3)  0          ['image[0][0]',                  
                                                                  'image[0][0]',                  
                                                                  'image[0][0]']                  
                 

In [9]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# 1. RESULT LOCATIONS
# ============================================================

RESULT_ROOT = Path(
    r"D:\chaitali\qgacef"
    r"\QGECAF_REVIEWER_REVISION_NESTED_BGWO"
)

ORIGINAL_LOCKED_MANIFEST = Path(
    r"D:\chaitali\qgacef"
    r"\QGECAF_FROM_SCRATCH_RUN_01"
    r"\locked_patient_fold_assignments.csv"
)

print("=" * 90)
print("QGECAF SAVED-RESULT VERIFICATION")
print("=" * 90)
print("Result root:", RESULT_ROOT)
print("Result root exists:", RESULT_ROOT.is_dir())

if not RESULT_ROOT.is_dir():
    raise FileNotFoundError(f"Result folder not found:\n{RESULT_ROOT}")

# ============================================================
# 2. VERIFY PRINCIPAL FILES
# ============================================================

principal_files = {
    "Fold-wise metrics":
        RESULT_ROOT / "fold_metrics.csv",

    "Cross-validation summary":
        RESULT_ROOT / "cross_validation_summary.csv",

    "Final test metrics":
        RESULT_ROOT / "final_test_metrics.csv",

    "Final model":
        RESULT_ROOT / "final_untouched_test" / "final_model.keras",

    "Final model summary":
        RESULT_ROOT / "final_untouched_test" / "final_model_summary.txt",

    "Final training history":
        RESULT_ROOT / "final_untouched_test" / "training_history.csv",

    "Final test predictions":
        RESULT_ROOT / "final_untouched_test"
        / "test_predictions_uncertainty.csv",

    "Final preprocessing":
        RESULT_ROOT / "final_untouched_test" / "preprocessing.joblib",
}

print("\n" + "=" * 90)
print("PRINCIPAL FILES")
print("=" * 90)

for description, path in principal_files.items():
    status = "FOUND" if path.is_file() else "NOT FOUND"
    print(f"{description:30s}: {status}")
    print(f"  {path}")

# ============================================================
# 3. FIND ALL PREDICTION FILES
# ============================================================

prediction_files = sorted(
    path for path in RESULT_ROOT.rglob("*")
    if path.is_file()
    and "prediction" in path.name.lower()
    and path.suffix.lower() in {".csv", ".xlsx", ".npz", ".npy"}
)

print("\n" + "=" * 90)
print("PREDICTION FILES")
print("=" * 90)

if prediction_files:
    for path in prediction_files:
        print(path)
else:
    print("No prediction files were found.")

# ============================================================
# 4. FIND PATIENT ASSIGNMENTS AND PARTITION AUDITS
# ============================================================

patient_assignment_files = sorted(
    path for path in RESULT_ROOT.rglob("*")
    if path.is_file()
    and any(
        keyword in path.name.lower()
        for keyword in [
            "patient",
            "manifest",
            "assignment",
            "partition",
            "split_audit",
        ]
    )
    and path.suffix.lower() in {".csv", ".xlsx", ".json"}
)

print("\n" + "=" * 90)
print("PATIENT-ASSIGNMENT AND PARTITION FILES")
print("=" * 90)

if patient_assignment_files:
    for path in patient_assignment_files:
        print(path)
else:
    print("No patient-assignment files were found inside the revised run.")

print("\nOriginal locked manifest:")
print(ORIGINAL_LOCKED_MANIFEST)
print("Exists:", ORIGINAL_LOCKED_MANIFEST.is_file())

# ============================================================
# 5. FIND FOLD DIRECTORIES
# ============================================================

fold_directories = sorted(
    path for path in RESULT_ROOT.rglob("*")
    if path.is_dir()
    and re.fullmatch(r"fold[_ -]?0?[1-9]|fold[_ -]?10",
                     path.name.lower())
)

print("\n" + "=" * 90)
print("FOLD DIRECTORIES")
print("=" * 90)

if fold_directories:
    for path in fold_directories:
        print(path)
else:
    print("No explicitly named fold directories were found.")

# ============================================================
# 6. VERIFY FOLD_METRICS.CSV
# ============================================================

fold_metrics_file = RESULT_ROOT / "fold_metrics.csv"

print("\n" + "=" * 90)
print("FOLD-METRICS VERIFICATION")
print("=" * 90)

if fold_metrics_file.is_file():
    fold_metrics = pd.read_csv(fold_metrics_file)

    print("File:", fold_metrics_file)
    print("Shape:", fold_metrics.shape)
    print("Columns:")
    print(fold_metrics.columns.tolist())

    fold_column = next(
        (
            column for column in fold_metrics.columns
            if column.lower() in {
                "fold",
                "outer_fold",
                "fold_number",
            }
        ),
        None
    )

    if fold_column is not None:
        folds = sorted(
            pd.to_numeric(
                fold_metrics[fold_column],
                errors="coerce"
            ).dropna().astype(int).unique().tolist()
        )

        print("\nFold column:", fold_column)
        print("Saved folds:", folds)
        print("Number of unique folds:", len(folds))

        if folds == list(range(1, 11)):
            print("PASS: All ten QGECAF outer-fold results are saved.")
        else:
            print("WARNING: Expected folds 1–10 but found:", folds)
    else:
        print("WARNING: A fold-number column was not identified.")

    display(fold_metrics.head(10))

else:
    print("fold_metrics.csv was not found.")

# ============================================================
# 7. VERIFY THE LOCKED PATIENT MANIFEST
# ============================================================

print("\n" + "=" * 90)
print("LOCKED PATIENT-FOLD MANIFEST")
print("=" * 90)

manifest_candidates = list(
    RESULT_ROOT.rglob("locked_patient_fold_assignments.csv")
)

if ORIGINAL_LOCKED_MANIFEST.is_file():
    manifest_candidates.append(ORIGINAL_LOCKED_MANIFEST)

# Remove duplicate paths
manifest_candidates = list(dict.fromkeys(manifest_candidates))

if not manifest_candidates:
    print("WARNING: No locked fold manifest was found.")
else:
    for manifest_path in manifest_candidates:
        print("\nManifest:", manifest_path)

        manifest = pd.read_csv(manifest_path)
        print("Shape:", manifest.shape)
        print("Columns:", manifest.columns.tolist())

        if "outer_fold" in manifest.columns:
            fold_values = sorted(
                pd.to_numeric(
                    manifest["outer_fold"],
                    errors="coerce"
                ).dropna().astype(int).unique().tolist()
            )

            print("outer_fold values:", fold_values)

            if set(range(1, 11)).issubset(set(fold_values)):
                print("PASS: Locked outer folds 1–10 are present.")

            if -1 in fold_values:
                print(
                    "PASS: The untouched final test cohort "
                    "(outer_fold = -1) is present."
                )

# ============================================================
# 8. FINAL DECISION
# ============================================================

fold_metrics_ready = fold_metrics_file.is_file()
manifest_ready = len(manifest_candidates) > 0
predictions_ready = len(prediction_files) > 0

print("\n" + "=" * 90)
print("READINESS FOR MOBILENETV2 COMPARISON")
print("=" * 90)
print("Fold-wise QGECAF metrics available:", fold_metrics_ready)
print("Locked patient assignments available:", manifest_ready)
print("Saved prediction files available:", predictions_ready)

if fold_metrics_ready and manifest_ready:
    print(
        "\nPASS: The existing QGECAF results can be retained for the "
        "matched MobileNetV2 comparison."
    )
    print(
        "You only need to train MobileNetV2 on the same locked "
        "patient-level folds."
    )
else:
    print(
        "\nThe required QGECAF comparison artifacts are incomplete. "
        "Review the missing items above before starting MobileNetV2."
    )

QGECAF SAVED-RESULT VERIFICATION
Result root: D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO
Result root exists: True

PRINCIPAL FILES
Fold-wise metrics             : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\fold_metrics.csv
Cross-validation summary      : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\cross_validation_summary.csv
Final test metrics            : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\final_test_metrics.csv
Final model                   : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\final_untouched_test\final_model.keras
Final model summary           : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\final_untouched_test\final_model_summary.txt
Final training history        : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\final_untouched_test\training_history.csv
Final test predictions        : FOUND
  D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_N

,accuracy,balanced_accuracy,macro_f1,weighted_f1,mcc,ece_15_bins,brier_multiclass,negative_log_likelihood,uncertainty_error_auroc,quality_uncertainty_spearman_rho,...,best_epoch,epochs_run,training_time_seconds,mean_epoch_time_seconds,inner_train_images,inner_validation_images,outer_holdout_images,inner_train_patients,inner_validation_patients,outer_holdout_patients
0,0.941642,0.950034,0.926788,0.942669,0.925088,0.036799,0.102015,0.258917,0.703363,-0.077679,...,10.0,18.0,645.287453,35.769922,9142.0,1112.0,1011.0,1288.0,166.0,158.0
1,0.931434,0.930121,0.916606,0.931776,0.913778,0.040154,0.105978,0.255506,0.834534,0.001544,...,11.0,19.0,636.597326,33.444445,9034.0,1108.0,1123.0,1292.0,156.0,164.0
2,0.904952,0.916753,0.885366,0.906760,0.881351,0.040532,0.147673,0.350356,0.803263,0.042453,...,8.0,16.0,581.225539,36.227537,8903.0,1110.0,1252.0,1286.0,158.0,168.0
3,0.926674,0.938278,0.915378,0.927252,0.908640,0.019224,0.120745,0.295868,0.705358,-0.078329,...,11.0,19.0,717.465603,37.595549,9239.0,1085.0,941.0,1292.0,157.0,163.0
4,0.902546,0.911469,0.891890,0.902188,0.879637,0.018672,0.157250,0.380133,0.790374,-0.078913,...,11.0,19.0,638.394759,33.530372,9050.0,1076.0,1139.0,1287.0,159.0,166.0
5,0.918216,0.936329,0.908401,0.918471,0.896792,0.025615,0.140686,0.343016,0.704275,-0.066740,...,16.0,24.0,806.871009,33.559806,9075.0,1114.0,1076.0,1296.0,159.0,157.0
6,0.941360,0.951684,0.928298,0.942136,0.923469,0.032278,0.099722,0.257383,0.737852,0.011108,...,11.0,19.0,719.791064,37.769913,8832.0,1154.0,1279.0,1290.0,161.0,161.0
7,0.920000,0.936135,0.903236,0.921491,0.899822,0.026929,0.122971,0.294549,0.677697,-0.062128,...,12.0,20.0,740.987767,36.978261,9172.0,993.0,1100.0,1293.0,162.0,157.0
8,0.933277,0.946885,0.919056,0.934288,0.916122,0.039725,0.112087,0.279386,0.785824,-0.086049,...,10.0,18.0,608.504931,33.739973,9013.0,1068.0,1184.0,1285.0,166.0,161.0
9,0.939655,0.941939,0.923934,0.940532,0.921768,0.018693,0.102167,0.265619,0.717693,0.014081,...,18.0,26.0,876.541066,33.655511,9146.0,959.0,1160.0,1300.0,155.0,157.0



LOCKED PATIENT-FOLD MANIFEST

Manifest: D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\locked_patient_fold_assignments.csv
Shape: (12400, 6)
Columns: ['path', 'patient_id', 'label', 'class_name', 'original_split', 'outer_fold']
outer_fold values: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PASS: Locked outer folds 1–10 are present.
PASS: The untouched final test cohort (outer_fold = -1) is present.

Manifest: D:\chaitali\qgacef\QGECAF_FROM_SCRATCH_RUN_01\locked_patient_fold_assignments.csv
Shape: (12400, 6)
Columns: ['path', 'patient_id', 'label', 'class_name', 'original_split', 'outer_fold']
outer_fold values: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PASS: Locked outer folds 1–10 are present.
PASS: The untouched final test cohort (outer_fold = -1) is present.

READINESS FOR MOBILENETV2 COMPARISON
Fold-wise QGECAF metrics available: True
Locked patient assignments available: True
Saved prediction files available: True

PASS: The existing QGECAF results can be retained for the matched Mobi